In [23]:
import pandas as pd
import json

In [47]:
dfs = []

In [48]:
xl = pd.ExcelFile('all_iso.xlsx')
assets = xl.sheet_names  # see all sheet names

In [49]:
for asset in assets:
    if asset == 'failure_codes':
        continue
    df = pd.read_excel('all_iso.xlsx', sheet_name=asset, header=1)
    df['asset_name'] = asset
    dfs.append(df)

In [75]:
metadata = {}

In [76]:
for i, df in enumerate(dfs):
    if assets[i] == 'failure_codes':
        continue
    for j in range(len(df)):
        row = df.iloc[j]
        cols = df.columns
        failure_code = row[cols[0]]
        if failure_code == '0TH':
            break
        desc = row[cols[1]]
        examples = row[cols[2]]
        components = []
        for col in df.columns[3:]:
            if row[col] == 'x':
                components.append(col)
        samples = {
            'sample': examples,
            'asset_name': assets[i],
            'components': components
        }
        if not failure_code in metadata:
            metadata[failure_code] = {
                'description': desc, 
                'examples': [samples]
            }
        else:
            metadata[failure_code]['examples'].append(samples)

In [ ]:
jsons = []

In [80]:
for key in metadata:
    item = {'label': key}
    for k in metadata[key]:
        item[k] = metadata[key][k]
    jsons.append(item)

In [81]:
with open('metadata.jsonl', 'w') as f:
    for item in jsons:
        json.dump(item, f)
        f.write('\n')